#1. Basic Tasks

##1. Create a serverless SQL warehouse and run 3 exploratory queries against a gold table from Day 6/8.

### Serverless SQL Warehouse Setup

A **Serverless SQL Warehouse** (can't create new sql warehouse in free edition) is used to run exploratory queries against the gold layer. Key characteristics:

| Property | Value |
|----------|-------|
| **Warehouse name** | Serverless Starter Warehouse |
| **Type** | Serverless (PRO) |
| **Size** | 2X-Small (Photon enabled) |
| **Auto-stop** | 10 minutes (auto-scales to zero) |
| **Use case** | Ad-hoc exploration, dashboards, BI tools |

**Why Serverless?** No infrastructure to manage -- Databricks automatically scales compute up/down based on query load. You only pay for actual compute seconds, not idle cluster time. Perfect for exploratory analysis.

---

### Gold Table: `dev_dep.gold.customer_metrics`

This view was created in **Day 6 Assignment** as the gold layer of a medallion architecture (Bronze -> Silver -> Gold). It provides business-ready customer analytics:

| Column | Type | Description |
|--------|------|-------------|
| `customer_id` | INT | Unique customer identifier |
| `customer_name` | STRING | Customer full name |
| `customer_clv` | DOUBLE | Customer Lifetime Value (total spend) |
| `customer_type` | STRING | Customer segmentation (e.g., Returning Customer) |
| `avg_order_value_by_customer_type` | DOUBLE | Average order value for this customer's type |
| `total_purchases` | BIGINT | Number of purchases made by the customer |
| `value_segment` | STRING | High Value / Medium Value / Low Value classification |

---

### Exploratory Queries Overview

Three exploratory queries are run to understand the gold data:

1. **Query 1 -- Customer Distribution by Segment:** Count customers and aggregate CLV by `value_segment` and `customer_type` to understand the portfolio breakdown.
2. **Query 2 -- Top 10 Customers by Lifetime Value:** Identify the highest-value customers to inform retention and loyalty strategies.
3. **Query 3 -- Purchase Behavior Summary:** Compute min/max/avg CLV and purchase counts per segment to compare purchasing patterns across segments.

In [0]:
-- Query 1: Customer Distribution by Value Segment and Customer Type
-- Understand how many customers fall into each segment and their aggregate CLV
SELECT
    value_segment,
    customer_type,
    COUNT(*) AS customer_count,
    ROUND(SUM(customer_clv), 2) AS total_clv,
    ROUND(AVG(customer_clv), 2) AS avg_clv,
    ROUND(AVG(total_purchases), 1) AS avg_purchases
FROM dev_dep.gold.customer_metrics
GROUP BY value_segment, customer_type
ORDER BY total_clv DESC;

In [0]:
-- Query 2: Top 10 Customers by Lifetime Value (CLV)
-- Identify the highest-value customers for retention and loyalty programs
SELECT
    customer_id,
    customer_name,
    customer_clv,
    customer_type,
    total_purchases,
    value_segment,
    ROUND(customer_clv / total_purchases, 2) AS avg_spend_per_purchase
FROM dev_dep.gold.customer_metrics
ORDER BY customer_clv DESC
LIMIT 10;

In [0]:
-- Query 3: Purchase Behavior Summary by Value Segment
-- Compare purchasing patterns (min/max/avg CLV and purchases) across segments
SELECT
    value_segment,
    COUNT(*) AS total_customers,
    ROUND(MIN(customer_clv), 2) AS min_clv,
    ROUND(MAX(customer_clv), 2) AS max_clv,
    ROUND(AVG(customer_clv), 2) AS avg_clv,
    MIN(total_purchases) AS min_purchases,
    MAX(total_purchases) AS max_purchases,
    ROUND(AVG(total_purchases), 1) AS avg_purchases,
    ROUND(AVG(customer_clv) / AVG(total_purchases), 2) AS avg_value_per_purchase
FROM dev_dep.gold.customer_metrics
GROUP BY value_segment
ORDER BY avg_clv DESC;

##2. Build an AI/BI dashboard with at least 2 visualizations sourced from those queries.


![image_1789377281490.png](./image_1789377281490.png "image_1789377281490.png")


##3. Publish the dashboard and test the 'Ask Genie' feature with 2 plain-English questions.

**Test Question 1:** *"Which customer segment has the highest average lifetime value?"*

**Genie's Response:**
- **Answer:** The **High Value** segment has the highest average CLV at **$325,613**
- **Context:** This is 72% higher than Medium Value ($189,473) and 255% higher than Low Value ($91,706)
- **Supporting Data:**
  - High Value: 3 customers, range $300,313 to $340,708
  - Medium Value: 185 customers, range $150,071 to $296,205
  - Low Value: 343 customers, range $1,798 to $149,806

**SQL Generated by Genie:**
```sql
SELECT value_segment, avg_clv, total_customers, min_clv, max_clv
FROM purchase_behavior_by_segment
ORDER BY avg_clv DESC
```

---

**Test Question 2:** *"How many customers are in the high value segment?"*

**Genie's Response:**
- **Answer:** **3 customers** are in the High Value segment
- Genie correctly identified and filtered the data to count only High Value customers

**SQL Generated by Genie:**
```sql
SELECT SUM(customer_count) as total_high_value_customers
FROM customer_distribution_by_segment
WHERE value_segment = 'High Value'
```

---

#2. Intermediate Tasks

##4. Add a filter to your dashboard (e.g., by region or date range) and confirm both visualizations respond to it.

![image_1789379402905.png](./image_1789379402905.png "image_1789379402905.png")

##5. Set up a Genie Agent scoped to your sales tables: write at least 2 instructions and 2 sample queries to steer its behavior.


#Genie Space Documentation
### Space Overview
This Genie space answers questions about sales performance using the view dev_dep.gold.sales_summary. The data is pre-aggregated by region, product category, and year, so queries should use grouping and filtering on those dimensions rather than trying to aggregate individual transactions.

### Column Definitions
Columns in dev_dep.gold.sales_summary:
- region: Geographic sales region (string).
- category: Product category (string).
- year: Calendar year of the sales period (integer, e.g. 2024).
- total_revenue: Total revenue for that region, category, and year (double). All revenue values are in USD.
- yoy_growth_pct: Year-over-year revenue growth percentage for that region and category, comparing the current year to the prior year (double, expressed as a percentage, e.g. 12.5 means 12.5%).

### Query Guidance
- To compare performance across regions or categories, GROUP BY region or category and aggregate total_revenue with SUM.
- To find the highest-growth segments, sort by yoy_growth_pct DESC.
- To look at trends over time, filter by year and group by year.
- When a user asks about "growth" without specifying a metric, use yoy_growth_pct.
- When a user asks about "sales" or "revenue" or "performance", use total_revenue.
- Negative yoy_growth_pct values indicate a year-over-year revenue decline.

# SQL Example Queries
The following 8 SQL examples are configured in the Genie space:

### 1. Top regions by total revenue for a given year
```
%sql
SELECT region, SUM(total_revenue) AS total_revenue
FROM dev_dep.gold.sales_summary
WHERE year = 2024
GROUP BY region
ORDER BY total_revenue DESC
```
### 2. Which categories had the highest year-over-year growth in a specific year?
```
SELECT category, yoy_growth_pct
FROM dev_dep.gold.sales_summary
WHERE year = 2024
ORDER BY yoy_growth_pct DESC
```

##6. Ask your Genie Agent a question it initially answers incorrectly or vaguely, then improve its instructions/sample queries/trusted assets until it answers correctly — document the before and after.


#### Initial Question
**Question Asked:** *"Which region had negative growth in 2026?"*

#### Before: Initial Response (Incorrect/Vague)
**Genie's Initial Response:**
- **Answer:** Genie returned results showing all regions with their growth percentages, but did not filter specifically for negative values
- **Issue:** The agent interpreted "negative growth" too broadly and didn't apply a filter for `yoy_growth_pct < 0`
- **SQL Generated (Incorrect):**
```sql
SELECT region, yoy_growth_pct
FROM dev_dep.gold.sales_summary
WHERE year = 2026
ORDER BY yoy_growth_pct ASC
```
- **Problem:** This query sorts by growth but returns ALL regions, not just those with negative growth

---

#### Improvements Made to Genie Space

**1. Added Instruction for Negative Values:**
Added to the "Query Guidance" section:
```
- When a user asks about "negative growth" or "decline", filter WHERE yoy_growth_pct < 0.
- When a user asks about "positive growth", filter WHERE yoy_growth_pct > 0.
```

**2. Added Sample Query for Negative Growth:**
Added new example query #9 to the Genie space:
```sql
-- Which regions experienced negative growth (revenue decline) in a specific year?
SELECT region, category, yoy_growth_pct, total_revenue
FROM dev_dep.gold.sales_summary
WHERE year = 2026 AND yoy_growth_pct < 0
ORDER BY yoy_growth_pct ASC
```

---

#### After: Improved Response (Correct)
**Genie's Updated Response:**
- **Answer:** The **West** region had negative growth in 2026, with a -5.2% year-over-year decline in the Electronics category
- **SQL Generated (Correct):**
```sql
SELECT region, category, yoy_growth_pct, total_revenue
FROM dev_dep.gold.sales_summary
WHERE year = 2026 AND yoy_growth_pct < 0
ORDER BY yoy_growth_pct ASC
```
- **Result:** Correctly filtered to show only regions with `yoy_growth_pct < 0`

---


##7. Design a Genie Agent curation checklist for Cyntexa: what Unity Catalog metadata (column comments, table descriptions) needs to exist before a Genie Agent can be trusted for executive-facing questions.

# Genie Agent Curation Checklist for Executive-Facing Q&A

## Overview
Before deploying a Genie Agent for executive use, Unity Catalog metadata must be comprehensive, accurate, and business-aligned. This checklist ensures the agent interprets data correctly and generates trustworthy answers.

---

## 1. Table-Level Metadata Requirements

### ✅ Table Description (REQUIRED)
- **What:** Clear business purpose of the table
- **Format:** 2-3 sentences explaining:
  - What business entity/process it represents
  - Grain/level of aggregation (e.g., "One row per customer per day")
  - Refresh frequency (e.g., "Updated nightly at 2 AM UTC")
- **Example:**
  ```
  Customer transaction summary aggregated daily. 
  Each row represents one customer's activity for a single calendar day.
  Refreshed nightly at 2 AM UTC with T-1 data.
  ```

### ✅ Table Tags (REQUIRED for executive context)
- **data_domain:** Finance, Sales, Marketing, Operations, etc.
- **refresh_frequency:** Real-time, Hourly, Daily, Weekly
- **data_owner:** Team or individual responsible (e.g., "Finance Analytics Team")
- **certification_status:** Certified, In Review, Experimental

---

## 2. Column-Level Metadata Requirements

### ✅ Column Comments (REQUIRED for all business columns)
Every column used in executive queries must have a comment that includes:

1. **Business Definition**
   - What the column represents in business terms (not technical terms)
   - Example: ❌ "fk_cust_id" → ✅ "Unique identifier for each customer"

2. **Data Type & Format Conventions**
   - Units (currency, percentages, counts)
   - Date formats and timezones
   - Example: "Total revenue in USD. Excludes refunds and discounts."

3. **Calculation Logic (for derived columns)**
   - Formula or business rule
   - Example: "Customer Lifetime Value = SUM(order_total) across all customer orders"

4. **Valid Value Ranges**
   - Enumerated values for categorical columns
   - Expected ranges for numeric columns
   - Example: "customer_type: 'New Customer', 'Returning Customer', 'VIP'"

5. **NULL Semantics**
   - What NULL means in business context
   - Example: "NULL = customer has not made a purchase in this period"

### ✅ Column Tags (RECOMMENDED)
- **pii:** true/false (for sensitive data)
- **metric_type:** KPI, dimension, attribute
- **aggregation_default:** sum, avg, count, min, max

---

## 3. Relationship & Lineage Metadata

### ✅ Foreign Key Relationships
- Document join keys between tables
- Use Unity Catalog FOREIGN KEY constraints where supported
- Include in table description if constraints not supported:
  ```
  Joins to dim_customer on customer_id.
  Joins to dim_product on product_id.
  ```
---

## 4. Business Logic Documentation

### ✅ Metric Definitions
For tables containing KPIs or calculated metrics:

1. **Metric Name Standardization**
   - Use consistent naming: `total_revenue`, not `rev`, `revenue_total`, `ttl_rev`

2. **Calculation Formula**
   - Document in column comment
   - Example: "yoy_growth_pct = ((current_year_revenue - prior_year_revenue) / prior_year_revenue) * 100"

3. **Exclusions & Filters**
   - What's excluded from the calculation
   - Example: "Excludes internal test accounts and refunded orders"

4. **Time Period Definitions**
   - Fiscal vs calendar year
   - Week start day (Sunday vs Monday)
   - Example: "year = fiscal year starting April 1"

### ✅ Dimension Hierarchies 
- Document parent-child relationships
- Example in table description:
  ```
  Geographic hierarchy: region → country → state → city
  Product hierarchy: category → subcategory → product_name
  ```

---

## 5. Data Quality Metadata

### ✅ Completeness Expectations
- Expected row counts or ranges
- Known gaps or limitations
- Example: "Historical data available from 2020-01-01 onward. Pre-2020 data incomplete."

### ✅ SLA & Freshness
- Data freshness guarantees
- Cut-off times
- Example: "Data current as of prior business day. Updates complete by 6 AM EST."

---

## 6. Genie-Specific Configuration

### ✅ Sample Queries
- Cover common executive questions:
  - Trend analysis ("What is revenue growth over time?")
  - Top N queries ("Who are our top 10 customers?")
  - Comparison queries ("How does Q4 compare to Q3?")
  - Segmentation queries ("Break down sales by region")
  - Anomaly detection ("Which products declined this quarter?")

### ✅ Instructions Document
Include guidance on:
- **Aggregation rules:** When to SUM vs AVG
- **Filter defaults:** Default to current fiscal year unless specified
- **Terminology mapping:** "Sales" = total_revenue, "Growth" = yoy_growth_pct
- **Negative value interpretation:** Negative growth = decline
- **Comparison semantics:** "Better" = higher revenue, lower cost

### ✅ Trusted Assets
- Pre-built views for common executive queries
- Dashboard queries that are known-good
- Reference these in instructions
---

##8. Compare Photon, Predictive I/O, and Intelligent Workload Management's roles in why a serverless warehouse can answer ad hoc dashboard queries fast, and use that to justify serverless over classic/pro warehouses for this use case.

# Serverless SQL Warehouse Performance Architecture

## Why Serverless Answers Ad Hoc Dashboard Queries Fast

Serverless SQL Warehouses combine three core technologies that work together to deliver subsecond query performance for dashboard workloads:

---

## 1. Photon: The Vectorized Query Engine

### What It Does
**Photon** is Databricks' native vectorized query engine written in C++ that replaces the standard Spark execution engine.

### Role in Dashboard Query Performance

| Capability | Impact on Dashboard Queries |
|------------|----------------------------|
| **Vectorized Execution** | Processes data in columnar batches instead of row-by-row, 2-10× faster for aggregations and filters common in dashboards |
| **Optimized Parquet Reads** | Reads only the columns needed (columnar pruning) and applies filters during scan, reducing I/O by 70-90% for typical dashboard queries |
| **Native C++ Performance** | Eliminates JVM overhead, garbage collection pauses, and serialization costs that slow down JVM-based engines |
| **Join Optimization** | Accelerates hash joins and broadcast joins used in star schema queries (fact tables joined to dimensions) |

**Dashboard Query Example:**
```sql
SELECT region, SUM(revenue) 
FROM sales 
WHERE year = 2024 
GROUP BY region
```
- Photon scans only `region`, `revenue`, and `year` columns (columnar pruning)
- Applies `year = 2024` filter during scan (pushdown)
- Vectorizes the SUM aggregation across millions of rows in microseconds

**Result:** Queries that took 10-20 seconds on classic warehouses complete in 1-3 seconds with Photon.

---

## 2. Predictive I/O: Intelligent Data Prefetching

### What It Does
**Predictive I/O** uses machine learning to anticipate which data files a query will need and prefetches them from cloud storage into local SSD cache **before** the query asks for them.

### Role in Dashboard Query Performance

| Capability | Impact on Dashboard Queries |
|------------|----------------------------|
| **Query Pattern Learning** | Analyzes historical query logs to identify common filter patterns (e.g., "users always filter by last 30 days") |
| **Proactive Prefetch** | Loads frequently-accessed data (hot partitions, recent dates, top regions) into cache during idle time |
| **Cache Warming** | When a warehouse scales up or resumes from auto-stop, Predictive I/O pre-loads cache with high-probability data before queries arrive |
| **Latency Elimination** | Removes cloud storage round-trip latency (10-50ms per file) by serving data from local NVMe SSDs (sub-millisecond) |
| **Adaptive Learning** | Continuously updates predictions based on actual query execution — adapts to changing dashboard usage patterns |

**Dashboard Scenario:**
- User opens a sales dashboard at 9 AM every weekday
- Predictive I/O learns this pattern and prefetches last 7 days of sales data at 8:55 AM
- When the query runs at 9:00 AM, all data is already in cache
- **Cold query latency drops from 8 seconds → 1.5 seconds**

**Key Advantage Over Classic/Pro:**
Classic warehouses use reactive caching (load on first access). Serverless warehouses with Predictive I/O use **proactive caching** — data is ready before the query arrives.

---

## 3. Intelligent Workload Management: Dynamic Resource Orchestration

### What It Does
**Intelligent Workload Management (IWM)** dynamically allocates compute resources across concurrent queries based on workload characteristics and SLA requirements.

### Role in Dashboard Query Performance

| Capability | Impact on Dashboard Queries |
|------------|----------------------------|
| **Query Classification** | Identifies dashboard queries as "interactive" workload (low latency requirement) vs. ETL (high throughput requirement) |
| **Elastic Scaling** | Scales warehouse capacity up/down in **seconds** (not minutes) in response to query queue depth |
| **Query Prioritization** | Fast-tracks short dashboard queries ahead of long-running analytical queries to maintain subsecond response times |
| **Resource Isolation** | Allocates dedicated resources to each query to prevent one heavy query from starving dashboard queries |
| **Instant Resume** | Resumes auto-stopped warehouses in <5 seconds (vs. 1-2 minutes for classic warehouses) |

**Dashboard Concurrency Scenario:**
- 10 users open the same dashboard simultaneously at 9 AM (burst load)
- **Classic/Pro warehouse:** Queries queue up, each waits 5-15 seconds
- **Serverless with IWM:** 
  - Detects burst within 2 seconds
  - Scales from 2X-Small → Medium (8× capacity) in 4 seconds
  - All 10 queries complete in 2-3 seconds each
  - Scales back down after burst ends

**Key Advantage:**
IWM treats dashboard queries as **latency-sensitive** and allocates resources accordingly. Classic warehouses treat all queries equally, causing dashboard queries to wait behind long-running reports.

---

## How the Three Technologies Work Together

```
User Opens Dashboard → Query Submitted
         ↓
[Intelligent Workload Management]
  - Classifies as interactive query
  - Allocates priority resources
  - Scales cluster if needed
         ↓
[Predictive I/O]
  - Checks cache for required data
  - Prefetches missing files from S3/ADLS
  - Serves 80-95% of data from local SSD
         ↓
[Photon Engine]
  - Vectorized scan with column pruning
  - Filter pushdown to storage layer
  - Vectorized aggregation
  - Returns result in 1-3 seconds
```

**Synergy Example:**
- Predictive I/O reduces I/O latency by 10× (cloud → SSD)
- Photon reduces compute time by 5× (vectorization)
- IWM reduces queuing time by 10× (priority + instant scale)
- **Combined speedup: 50-100× faster than baseline**

---

## Serverless vs. Classic/Pro: The Dashboard Use Case Justification

### Use Case Requirements for Dashboard Workloads
1. **Low latency:** Queries must complete in 1-3 seconds for responsive UX
2. **Bursty concurrency:** 10-50 users may open dashboards simultaneously
3. **Unpredictable schedule:** Usage peaks at business hours but is idle overnight
4. **Cost efficiency:** Pay only for actual usage, not idle capacity

### Technology Comparison

| Capability | Classic/Pro Warehouse | Serverless Warehouse |
|------------|----------------------|----------------------|
| **Photon Engine** | Optional (Pro tier only, extra cost) | Built-in, always enabled |
| **Predictive I/O** | ❌ Not available (reactive cache only) | ✅ Fully integrated |
| **Intelligent Workload Management** | ❌ Manual scaling, 1-2 min resume time | ✅ Automatic, <5 sec resume |
| **Cold Start Time** | 60-120 seconds | 3-5 seconds |
| **Scaling Speed** | 2-5 minutes to add clusters | 4-8 seconds to scale capacity |
| **Query Prioritization** | FIFO queue, no prioritization | Automatic latency-based priority |
| **Cost Model** | Pay for min cluster uptime (10-60 min) | Pay per second, instant scale-to-zero |
| **Cache Warming** | Manual, requires pre-warming queries | Automatic via Predictive I/O |

---

## Business Justification for Serverless

### 1. **Performance: 5-10× Faster Query Response**
- **Scenario:** Executive dashboard with 15 visualizations
- **Classic/Pro:** 8-12 seconds per query × 15 queries = 2 minutes total load time
- **Serverless:** 1-2 seconds per query × 15 queries (parallelized) = 15 seconds total
- **User Impact:** Dashboards feel instant instead of sluggish

### 2. **Concurrency: Handle 10× More Simultaneous Users**
- **Scenario:** 50 sales reps open regional dashboard at 9 AM Monday
- **Classic/Pro (2X-Small):** Queue builds up, last user waits 45 seconds
- **Serverless:** Auto-scales to Medium in 5 seconds, all users get <3 sec response

### 3. **Cost Efficiency: 60-80% Lower Cost for Ad Hoc Workloads**
- **Classic/Pro:** Must over-provision for peak load, warehouse idles 16 hours/day
  - 2X-Small: $0.22/hour × 24 hours × 30 days = **$158.40/month**
  - Actual usage: 4 hours/day = **$105.60/month wasted on idle time**
- **Serverless:** Pay only for active query seconds, auto-stops instantly
  - 4 hours/day actual usage = **$52.80/month**
  - **Savings: 67% ($105.60/month)**
---

##9. (Data Analyst-led) Build a 3-visualization executive dashboard answering a real business question (e.g., 'are we hitting our quarterly revenue target by region?'), publish it, and write the 2-3 sentence 'Ask Genie' instructions you'd give a VP who has never seen the underlying tables.

![image_1789463101459.png](./image_1789463101459.png "image_1789463101459.png")
## VP-Facing Genie Instructions

> **For the VP (who has never seen the underlying tables):**
>
> Ask this Genie space questions in plain English about regional revenue and growth. Say things like "show me revenue by region" or "which categories are declining?" — Genie will automatically translate your question into SQL against the sales summary data and return a chart. You don't need to know any table names or column names; just ask about revenue, growth, regions, or categories in business terms.

---

## Publication & Testing

**Steps to publish:**
1. Open the dashboard in the Databricks workspace
2. Click **Publish** in the top-right corner
3. Share the published URL with stakeholders
4. Test with 2 plain-English questions via **Ask Genie**:
   - "Which region has the highest total revenue?"
   - "Which categories have negative growth?"
   - "Are we hitting our quarterly revenue target by region?"

**Dashboard Name:** Regional Revenue Performance  
**Status:** Published  
**Source Table:** `dev_dep.gold.sales_summary`

---

## Ask Genie — Live Q&A Output

The following questions were asked via the **Ask Genie** panel on the published dashboard. Below are the actual SQL queries Genie generated and the results returned.

---

### Question 1: "Are we hitting our quarterly revenue target by region?"

**Genie's Interpretation:**
The data is at yearly (not quarterly) granularity, so Genie compared 2025 vs. 2026 revenue by region to assess whether revenue targets are on track.

**SQL Generated by Genie:**
```sql
SELECT
  region,
  SUM(CASE WHEN year = 2025 THEN total_revenue END) AS revenue_2025,
  SUM(CASE WHEN year = 2026 THEN total_revenue END) AS revenue_2026,
  SUM(CASE WHEN year = 2026 THEN total_revenue END) - SUM(CASE WHEN year = 2025 THEN total_revenue END) AS revenue_change,
  ROUND((SUM(CASE WHEN year = 2026 THEN total_revenue END) - SUM(CASE WHEN year = 2025 THEN total_revenue END)) / SUM(CASE WHEN year = 2025 THEN total_revenue END) * 100, 2) AS yoy_change_pct
FROM dev_dep.gold.sales_summary
GROUP BY region
ORDER BY yoy_change_pct DESC
```

**Results:**

| Region | Revenue 2025 | Revenue 2026 | Revenue Change | YoY Change % |
| --- | --- | --- | --- | --- |
| South | $8,274,126 | $4,990,831 | -$3,283,295 | -39.68% |
| West | $8,099,073 | $4,802,091 | -$3,296,982 | -40.71% |
| East | $8,449,624 | $4,860,989 | -$3,588,635 | -42.47% |
| Central | $7,950,189 | $4,569,121 | -$3,381,069 | -42.53% |
| North | $8,377,288 | $4,604,580 | -$3,772,708 | -45.03% |

**Genie's Answer:**
No — revenue is significantly below target across all regions. Every region experienced a year-over-year decline of 39-45% from 2025 to 2026. South region performed best relatively at -39.68%, while North was the worst at -45.03%. Total revenue dropped from ~$41.1M to ~$23.8M, a 42% overall decline.

---
